**Importing necessary libraries**

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from scipy import stats
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

**Importing the dataset to a dataframe**

In [ ]:
file_path = r'data_wrangling\final_data\2025_data.csv'
df = pd.read_csv(file_path, index_col=0)

**Sample of the dataframe**

In [ ]:
df.sample(5)

## **Univariate Analysis** 

**Numerical Variables**

In [ ]:
df['fuel'] = df['fuel'].replace('hybride', 'hybride essence')

*setting a permament categorical dataframe for categorical analysis*

In [ ]:
cat_df = df.select_dtypes(include=['object'])
cat_df = cat_df.drop(columns=['circulation-date','publish-date','model', 'location','brand'])

*Setting a permanent numerical dataframe for numerical analysis*

In [ ]:
num_df = df.select_dtypes(include='number')

In [ ]:
df.info()

In [ ]:
df.describe()

* *A general overview on each variable*

In [ ]:
fig, axes = plt.subplots(1,len(num_df.columns), figsize=(24,6))
for i, col in enumerate(num_df.columns):
    sns.boxplot(y=num_df[col], ax = axes[i], color='royalblue')
    axes[i].set_title(col)

plt.suptitle("Boxplot for numerical values", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:

fig, axes = plt.subplots(1, len(num_df.columns), figsize=(24,6))
for i,col in enumerate(num_df.columns):
    sns.histplot(num_df[col], ax = axes[i],bins=40, color='royalblue')
    axes[i].set_title(col)
plt.suptitle('Histograms for numerical values')
plt.tight_layout()
plt.show()

* The dataset exhibits right skewness across most variables, reflecting the nature of the used car market. The Year variable is an exception, indicating that most listed cars are relatively new. The Engine Size distribution is multimodal, representing the most common engine categories on the market, while other variables display a few high-end outliers.
* Consistent outliers appear across all boxplots, which is reasonable given the nature of the used car market, while most prices fall within the **36,000–85,000 TND** range.
* *Fiscal-Power* shows bi-modal distribution, highlighting the two most common values in the market, **4 CV** and **6 CV**.

**Categorical Variables**

*A general overview on each variable*

In [ ]:
fig, axes = plt.subplots(1,len(cat_df.columns)+1, figsize=(25,6))

sns.barplot(data=df['brand'].value_counts().head(12), ax=axes[0])
axes[0].set_title('brand')
axes[0].tick_params(axis='x', rotation=45)
for i, col in enumerate(cat_df.columns, start=1):
    sns.barplot(data=cat_df[col].value_counts(), ax=axes[i])
    axes[i].set_title(col)
    axes[i].tick_params(axis='x', rotation=45)
plt.suptitle('Bar Charts for categorical variables')
plt.tight_layout()
plt.show()

* Economic brands such as *Peugeot*, *Kia*, *Renault*, and *Volkswagen* are the most common. Nearly two-thirds of cars are manual, and the dominant body types—berline and compacte—reflect the purchasing power of Tunisian consumers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15,6))

pie_df = df['location'].value_counts().head(7)
explode = [0.1 if i < 3 else 0 for i in range(len(pie_df))]
colors = plt.cm.tab20.colors

axes[0].pie(pie_df.values, labels=pie_df.index, autopct='%1.1f%%', 
            startangle=90, explode=explode, colors=colors[:len(pie_df)], shadow=True)
axes[0].set_title('Top 7 locations')

top_models = df['model'].value_counts().head(15)
sns.barplot(x=top_models.index, y=top_models.values, ax=axes[1], palette='tab20')
axes[1].set_title('Top 15 models in sale')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

* Most cars are sold in the Greater Tunis area, with *Tunis*, *Ariana*, and *Ben Arous* together accounting for over half of all listings. Models like the *Golf*, *Series 3*, *Clio*, and *Rio*—mainly berlines and compacts—dominate the market.

*The Most Frequent Category From Each Column*

In [ ]:
frequency = df[['location','brand','model','fuel','gear','body-type']]
for i, category in enumerate(frequency.columns):
    print(f'{i}.The mode of the column {category}, is: {frequency[category].mode()}')

## **Bivariate Analysis**

In [ ]:
fig = plt.figure(figsize=(9,7))
correlation_matrix = df.corr(numeric_only=True)
sns.heatmap(correlation_matrix, cmap='coolwarm', annot=True, fmt=".2f", square=True, linewidths=.5)
plt.show()

* *Price* and *fiscal power* are highly correlated, indicating a strong relationship.

* *Mileage* and *car age* have the highest correlation.

* *Other variables* show moderate to weak correlations.

##### *Scatter plots were created for variable pairs with high correlation (greater than 0.5)*

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(13,5))

sns.regplot(data=df, x='price', y='fiscal-power',color='royalblue', ax=axes[0])
plt.xlabel('Price')
plt.ylabel('Fiscal Power (CV)')
axes[0].set_title('The relationship between price and fiscal power')

sns.regplot(data=df[(df['price']<=86000) & (df['fiscal-power']<12)], x='price', y='fiscal-power', color='royalblue', ax=axes[1]) #prices under the 3/4 IQR
plt.xlabel('Price')
plt.ylabel('Fiscal Power (CV)')
axes[1].set_title('The relationship between price and fiscal power (price extends to 3/4 IQR)')
plt.suptitle('Price Vs Fiscal Power')
plt.show()

* In the dense area, the correlation is clear and moderate. The price increases as the Fiscal power increases

In [ ]:
perm = df[((df['mileage']<500000)) & ((df['price']<=300000))] 
sns.scatterplot(data=perm, x='price', y='mileage', color='royalblue')
plt.show()

In [ ]:
sns.regplot(data=df, x='mileage', y='car-age', color='royalblue')
plt.show()

* Car age and mileage are highly correlated.

*Scatter plot of engine size versus fiscal power* 

In [ ]:
sns.scatterplot(data=df, x='engine-size', y='fiscal-power', color='royalblue')
plt.show()

* The scatter plot shows a positive correlation between engine size and fiscal power, with data concentrated between 1000-3000 engine size and 5-20 fiscal power, and outliers up to 6000 engine size.

*Removed suspicious and non-relevant rows.* 

In [ ]:
suspicious = df[
    ((df['engine-size']>=3000) & (df['fiscal-power']<11)) |
    ((df['engine-size']<=2000) & (df['fiscal-power']>12))
]
suspicious.shape[0]

df = df.drop(suspicious.index)

suspicious_b = df[df['fiscal-power'] == 0]
df = df.drop(suspicious_b.index)

* Noisy data points have been removed.

* *Strong correlation* exists between the two features, with the densest cluster between **1000** and **3000cc**, reflecting the most common cars in the market.

##### *Average prices by fuel type*

In [ ]:
perm = df.groupby('fuel')[['price']].mean().sort_values(by='price')
sns.barplot(data=perm, x='fuel', y='price', color='royalblue')
plt.xticks(rotation=45)
plt.ylabel('average price')

* *Diesel* car prices are slightly higher than *petrol*, while *hybrid cars* are the most expensive due to their novelty and higher costs.

*Average car prices by location*

In [ ]:
perm = df.groupby('location')[['price']].mean().sort_values(by='price', ascending=False).head(5)
sns.barplot(data=perm, x='location', y='price')

* Although *Kairouan* has a small market share (<2%), it ranks second in average car prices, indicating a niche high-value market *dominated by commercial vehicles and pickups.*

*Fuel Vs. Car Mileage*

In [ ]:
temp = df.groupby('fuel')[['mileage']].mean()
sns.boxplot(data=df, x='fuel', y='mileage', color='royalblue')
plt.xticks(rotation=45)

* *Diesel* cars have the highest mileage, with *hybrid diesel* engines also exceeding *hybrid petrol engines*.

* *Electric* cars have the lowest mileage and no outliers, reflecting their *recent introduction* to the Tunisian market.

*Most frequent car brands by gearbox type*

In [ ]:
temp = (
    df.groupby(['brand', 'gear']).size().unstack(fill_value=0).assign(total=lambda x: x.sum(axis=1)).sort_values(by='total', ascending=False).drop(columns='total').head(5))

temp.plot(kind='bar', stacked=True, figsize=(8,5))
plt.title("Gear Type Distribution by Brand")
plt.xlabel("Brand")
plt.ylabel("Count")
plt.legend(title="Gear")
plt.show()

* *Manual* gearboxes dominate low- to mid-priced cars, while premium brands like Mercedes-Benz use *automatic* gearboxes for nearly all their models.

*Comparison of the most expensive and least expensive car brands.*

In [ ]:
temp1 = df.groupby('brand').agg(price=('price','mean'),frequency=('price','count')).sort_values(by='price', ascending=False).head(5)
temp2 = df.groupby('brand')[['price']].mean().sort_values(by='price', ascending=True).head(5)

fig, ax = plt.subplots(figsize=(10,6))

sns.barplot(
    data=temp1.reset_index(),
    x='brand', y='price',
    color='royalblue', ax=ax
)

sns.barplot(
    data=temp2.reset_index(),
    x='brand', y='price',
    color='red', ax=ax
)

for container in ax.containers[len(temp1):]:
    for bar in container:
        bar.set_height(-bar.get_height())

ax.set_title("Most Expensive vs Cheapest Brands", fontsize=14)
ax.set_ylabel("Average Price")
ax.set_xlabel("Brand")
ax.axhline(0, color="black", linewidth=0.8)
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

* Premium brands like *Porsche* and *Land Rover* record the highest average prices.
* *Chinese* and *Indian* brands have the lowest average prices, reflecting their focus on low-cost vehicles.

*Distribution of car ages based on the year column*

In [ ]:
fig = plt.figure(figsize=(10,5))
temp = df.groupby('year').agg(frequency=('year','count')).sort_values(by='year')
sns.barplot(data=temp, x='year',y='frequency')
plt.xticks(rotation=45)
plt.xlabel('Year')
plt.ylabel('Number of cars')
plt.title('Car registration year distribution')

* Most cars sold are *under 16 years old*, with a notable peak in **2021**.

* 2020 shows a sharp drop in registrations, likely due to severe economic conditions caused by **COVID-19** or the Tunisian tendency to sell cars after holding them for at least five years

* **The peak in 2021** may indicate that consumer car purchases were higher than in previous years, suggesting a potential economic recovery following the *COVID-19* crisis.

## **Multivariate Analysis** 

*Most common body type sold per location and its average price*

In [ ]:
temp = df.pivot_table(values='model',index='body-type',columns='location',fill_value=0, aggfunc='count')
temp.index.name = 'body-type'
temp = temp.idxmax().reset_index()
temp.columns = ['location', 'most_frequent_bodytype']

locations = ['tunis','sousse','médenine','sidi bouzid','kairouan','gafsa']

temp1 = temp[temp['location'].isin(locations)]
temp2 = df.groupby(['location','body-type'])['price'].mean().reset_index()

temp = temp1.merge(temp2, left_on=['location','most_frequent_bodytype'],
                   right_on=['location','body-type'],
                   how='left')
temp = temp.drop(columns='body-type').rename(columns={'price':'avg_price'})

temp[['avg_price']] = round(temp[['avg_price']])

sns.barplot(data=temp, x='location',y='avg_price', hue='most_frequent_bodytype')
plt.show()

*Relationship between engine size and fiscal power, grouped by fuel type*

In [ ]:
sns.scatterplot(data=df, x='engine-size', y='fiscal-power',hue='fuel' , color='royalblue', alpha=0.3)
plt.show()

* *Diesel* engines tend to have **larger engine sizes** and **higher fiscal power** than *petrol* engines.

* *Electric* cars have zero fiscal power values, as expected.

*Distribution of fuel types in the five most common locations, including mileage*

In [ ]:
top_locations = ["sfax","sousse","ben arous","ariana","tunis"]
top_fuel = ["essence","diesel","hybride essence"]
temp = df[df['location'].isin(top_locations) & df['fuel'].isin(top_fuel)]
temp = temp[['location','fuel','mileage']]

plt.figure(figsize=(10,4))
sns.barplot(data=temp, x='location', y='mileage', hue ='fuel', estimator='mean', errorbar=None, palette=["#74C3FF", "#104D92", "#a037aa"])
plt.title('Average Mileage by Fuel Type in Top 5 Locations', fontsize=12)
plt.xlabel('Location', fontsize=10)
plt.ylabel('Mileage (KM)', fontsize=10)
plt.legend(title='Fuel Type', loc='upper right')
plt.show()

* *Diesel cars* with the highest average mileage across ben arous, sousse, ariana, tunis, and sfax, while *Patrol* and *Hybrid Patrol* vary, peaking at ben arous and sfax.

*Distribution of fuel types in the five most common locations, including price*

In [ ]:
top_locations = ["sfax","sousse","ben arous","ariana","tunis"]
top_fuel = ["essence","diesel","hybride essence"]
temp = df[df['location'].isin(top_locations) & df['fuel'].isin(top_fuel)]
temp = temp[['location','fuel','price']]

plt.figure(figsize=(10,4))
sns.barplot(data=temp, x='location', y='price', hue ='fuel', estimator='mean', errorbar=None, palette=["#74C3FF", "#104D92", "#a037aa"])
plt.title('Average Mileage by Fuel Type in Top 5 Locations', fontsize=12)
plt.xlabel('Location', fontsize=10)
plt.ylabel('Price (TND)', fontsize=10)
plt.legend(title='Fuel Type', loc='upper right')
plt.show()

* *Patrol Hybrid'* type has the highest average price across Ben Arous, Sousse, Ariana, Tunis, and Sfax, with *Diesel* showing moderate prices and *Patrol* the lowest, peaking in Tunis at around 175,000 TND.

In [ ]:
import plotly.express as px

fig = px.scatter_3d(
    df,
    x='engine-size',
    y='fiscal-power',
    z='price',
    color='price',
    size='price',
    color_continuous_scale='viridis',
    title='Engine Size vs Fiscal Power vs Price (Interactive 3D Plot)'
)

fig.update_layout(scene=dict(
    xaxis_title='Engine Size',
    yaxis_title='Fiscal Power',
    zaxis_title='Price'
))

fig.show()

